# Chapter 7 — ACPI on ARM: Tables, MADT, GTDT, IORT

## Concept
ARM platforms use ACPI (ACPI 6.x) for hardware description. Key tables:
RSDP (Root System Description Pointer), XSDT (extended root), DSDT
(Differentiated System Description), MADT (Multiple APIC Description Table —
encodes GICv3 distributor, redistributors, ITS), GTDT (Generic Timer
Description Table — ARM generic timer), IORT (I/O Remapping Table — SMMU,
ITS link). QEMU generates all tables via the `fw_cfg` interface.

## Lab Objectives
1. Locate RSDP at its expected address via `/sys/firmware/acpi/`.
2. Dump and parse MADT — confirm GICv3 distributor entry present.
3. Dump GTDT — confirm ARM generic timer entries.
4. Dump IORT — confirm SMMUv3 node when launched with `iommu=smmuv3`.

> **QEMU Fidelity Note** — Results from this lab are functionally valid on
> QEMU `virt` machine with HVF (macOS Apple Silicon). Behaviours that differ
> from real Neoverse silicon are annotated inline. See Section 6 of the
> project plan for the full gap table.


In [ ]:
import sys, os, pathlib, time
from IPython.display import display, HTML

# ── USER CONFIGURATION — edit these paths before running ──────────────────────
LABS_ROOT    = pathlib.Path.home() / "arm_qemu_labs"
SHARED_DIR   = LABS_ROOT / "shared"
DISK_IMAGE   = LABS_ROOT / "images" / "ubuntu-24.04-arm64.qcow2"
SEED_ISO     = LABS_ROOT / "images" / "seed.iso"   # cloud-init seed
FIRMWARE     = LABS_ROOT / "firmware" / "QEMU_EFI.fd"
CONSOLE_USER = "ubuntu"
CONSOLE_PASS = "arm-lab-2026"
CPU          = "cortex-a76"
RAM          = "2G"
SMP          = 1
# ─────────────────────────────────────────────────────────────────────────────

sys.path.insert(0, str(SHARED_DIR))
from qemu_launcher  import QEMULauncher
from qmp_client     import QMPClient
from serial_console import SerialConsole
from assert_lib     import (assert_true, assert_false, assert_equal,
                             assert_contains, assert_not_contains,
                             assert_qmp_ok, assert_in_range, summary, reset)
reset()

import shutil
assert shutil.which("qemu-system-aarch64"), (
    "FATAL: qemu-system-aarch64 not found in PATH. Run setup_qemu_labs.sh.")
assert FIRMWARE.exists(),   f"FATAL: Firmware not found at {FIRMWARE}"
assert DISK_IMAGE.exists(), f"FATAL: Disk image not found at {DISK_IMAGE}"
assert SEED_ISO.exists(),   f"FATAL: Seed ISO not found at {SEED_ISO}"
print("✓ Environment check passed")
print(f"  qemu-system-aarch64 : {shutil.which('qemu-system-aarch64')}")
print(f"  Firmware            : {FIRMWARE}")
print(f"  Disk image          : {DISK_IMAGE}")


In [ ]:
launcher = QEMULauncher(
    cpu=CPU, ram=RAM, smp=SMP,
    firmware=str(FIRMWARE),
    disk_image=str(DISK_IMAGE),
    seed_iso=str(SEED_ISO),
    extra_args=[],
)
launcher.launch()
launcher.wait_ready(timeout=15)
print(f"QEMU running — QMP port {launcher.qmp_port}, serial port {launcher.serial_port}")


In [ ]:
qmp = QMPClient(port=launcher.qmp_port)
qmp.connect(timeout=30)

sc = SerialConsole(port=launcher.serial_port)
sc.connect(timeout=30)

print("Waiting for guest boot (up to 120 s on HVF, longer on TCG)…")
sc.wait_for_boot(timeout=180)
sc.login(CONSOLE_USER, CONSOLE_PASS)
print("Guest ready.")


In [ ]:
# ── Step 1: Verify ACPI tables via /sys/firmware/acpi/tables ─────────────────
acpi_tables = sc.run_command(
    "ls /sys/firmware/acpi/tables/ 2>/dev/null || echo 'no ACPI tables'",
    timeout=10
)
print("ACPI tables:\n", acpi_tables)


In [ ]:
# ── Step 2: Read MADT table size and hex dump first bytes ────────────────────
madt_check = sc.run_command(
    "ls -la /sys/firmware/acpi/tables/APIC 2>/dev/null || "
    "ls -la /sys/firmware/acpi/tables/MADT 2>/dev/null || "
    "echo 'MADT not found (table name may be APIC)'",
    timeout=10
)
print(madt_check)

# Try xxd on MADT (ACPI uses 'APIC' as the MADT signature)
madt_hex = sc.run_command(
    "xxd /sys/firmware/acpi/tables/APIC 2>/dev/null | head -20 || "
    "xxd /sys/firmware/acpi/tables/MADT 2>/dev/null | head -20 || "
    "echo 'xxd not available or table absent'",
    timeout=10
)
print("MADT hex head:\n", madt_hex)


In [ ]:
# ── Step 3: Use acpidump/acpixtract if available ──────────────────────────────
acpidump_check = sc.run_command(
    "which acpidump 2>/dev/null && echo 'found' || echo 'not found'",
    timeout=5
)
print("acpidump:", acpidump_check.strip())

if "found" in acpidump_check:
    # Dump MADT (signature APIC in ACPI spec)
    madt_parsed = sc.run_command(
        "sudo acpidump -n APIC 2>/dev/null | head -80 || "
        "acpidump -n APIC 2>/dev/null | head -80 || echo 'acpidump APIC failed'",
        timeout=20
    )
    print("MADT dump:\n", madt_parsed[:2000])
else:
    # Fallback: grep GIC strings from binary table
    madt_parsed = sc.run_command(
        "strings /sys/firmware/acpi/tables/APIC 2>/dev/null | head -20 || "
        "echo 'strings not available'",
        timeout=10
    )
    print("MADT strings:", madt_parsed)


In [ ]:
# ── Step 4: Read GTDT (Generic Timer Description Table) ──────────────────────
gtdt_hex = sc.run_command(
    "xxd /sys/firmware/acpi/tables/GTDT 2>/dev/null | head -20 || "
    "echo 'GTDT not found'",
    timeout=10
)
print("GTDT hex head:\n", gtdt_hex)

# GTDT signature check: first 4 bytes should be "GTDT"
gtdt_sig = sc.run_command(
    "dd if=/sys/firmware/acpi/tables/GTDT bs=4 count=1 2>/dev/null | cat || "
    "echo 'GTDT read failed'",
    timeout=10
)
print("GTDT signature raw:", repr(gtdt_sig))


In [ ]:
# ── Step 5: Check IORT table (present when -machine iommu=smmuv3) ────────────
iort_check = sc.run_command(
    "ls /sys/firmware/acpi/tables/IORT 2>/dev/null && echo 'IORT present' || "
    "echo 'IORT absent (expected without -machine iommu=smmuv3)'",
    timeout=10
)
print("IORT:", iort_check.strip())


In [ ]:
# ── Step 6: Confirm ARM generic timer in dmesg ───────────────────────────────
timer_dmesg = sc.run_command(
    "dmesg | grep -i 'arch_timer\|generic timer\|GTDT\|armv8-timer'",
    timeout=10
)
print("Timer dmesg lines:\n", timer_dmesg)


In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert_contains(acpi_tables, r"APIC|MADT",
                "MADT (APIC) table present in /sys/firmware/acpi/tables",
                action="UEFI not providing ACPI; check QEMU_EFI.fd firmware")

assert_contains(acpi_tables, r"GTDT",
                "GTDT table present in /sys/firmware/acpi/tables",
                action="Generic timer ACPI table absent — check UEFI firmware")

assert_contains(acpi_tables, r"DSDT|FACP",
                "DSDT/FACP root table present",
                action="UEFI ACPI subsystem not active")

# MADT must contain GIC reference
madt_all = madt_hex + madt_parsed
assert_contains(madt_all, r"GIC|GICD|0x08000000|apic",
                "MADT contains GICv3 distributor reference",
                action="Check QEMU -machine virt GICv3 configuration")

# GTDT: ARM generic timer
assert_contains(timer_dmesg, r"arch_timer|armv8-timer|generic.timer",
                "ARM generic timer registered in dmesg",
                action="Check GTDT table and kernel CONFIG_ARM_ARCH_TIMER")

# IORT informational (only present with SMMU)
iort_present = "present" in iort_check
assert_true(True,   # informational — not required without SMMU
            f"IORT table: {'present' if iort_present else 'absent (correct without SMMU)'}",
            detail=iort_check.strip())


In [ ]:
# ── Teardown: always runs even if earlier cells raised ────────────────────────
try:
    qmp.quit()
except Exception:
    pass
try:
    sc.close()
except Exception:
    pass
try:
    launcher.terminate()
except Exception:
    pass
print("Teardown complete. QEMU process terminated.")


## Lab Summary

| Assertion | Expected | Notes |
|-----------|----------|-------|
| MADT (APIC) in sysfs | Present | UEFI ACPI generation |
| GTDT in sysfs | Present | Generic timer table |
| DSDT/FACP present | Present | ACPI root table |
| MADT GICv3 reference | Present | Distributor described |
| ARM generic timer in dmesg | Present | arch_timer driver |
| IORT present | Optional | Only with -machine iommu=smmuv3 |

> **Fidelity note**: QEMU generates ACPI tables via `fw_cfg`; the structure
> is identical to BIOS-built tables on real Neoverse. Content differs:
> GIC MMIO bases, timer frequencies, and PCI range descriptors reflect
> QEMU `virt` machine layout, not production board values.

## Next Steps
→ **Chapter 8**: Device Tree — extract and decompile the QEMU-generated DTB.
